In [1]:
import findspark
findspark.init()

In [2]:
# PySpark is the Spark API for Python. We are using PySpark to initialize the spark context. 
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

- SparkContext is the entry point for Spark applications and contains functions to create RDDs such as parallelize(). 
- SparkSession is needed for SparkSQL and DataFrame operations.

In [3]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 11:01:42 WARN Utils: Your hostname, Sayalis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.11 instead (on interface en0)
26/09/21 11:01:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 11:01:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/21 11:01:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


- To work with dataframes we just need to verify that the spark session instance has been created.

In [4]:
if 'spark' in locals() and isinstance(spark, SparkSession):
    print("SparkSession is active and ready to use.")
else:
    print("SparkSession is not active. Please create a SparkSession.")

SparkSession is active and ready to use.


- Create an RDD here by calling sc.parallelize()
- We create an RDD which has integers from 1 to 30.

In [8]:
data = range(1,30)
# print first element of iterator
print(data[0])
len(data)
xrangeRDD = sc.parallelize(data, 4)

# this will let us know that we created an RDD
xrangeRDD

1


PythonRDD[1] at RDD at PythonRDD.scala:59

##### Task 1: Create an RDD

- The `glom().collect()` one is nice for actually seeing the partitioning.

In [14]:
xrangeRDD.glom().collect()

[[1, 2, 3, 4, 5, 6, 7],
 [8, 9, 10, 11, 12, 13, 14],
 [15, 16, 17, 18, 19, 20, 21],
 [22, 23, 24, 25, 26, 27, 28, 29]]

##### Task 2: Transformations

- A transformation is an operation on an RDD that results in a new RDD. 
- The transformed RDD is generated rapidly because the new RDD is lazily evaluated, which means that the calculation is not carried out when the new RDD is generated. 
- The RDD will contain a series of transformations, or computation instructions, that will only be carried out when an action is called. 

In [15]:
subRDD = xrangeRDD.map(lambda x: x-1)
filteredRDD = subRDD.filter(lambda x : x<10)

##### Task 3: Actions

- A transformation returns a result to the driver. 
- collect() action to get the output from the transformation.




In [16]:
print(filteredRDD.collect())
filteredRDD.count()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


10

##### Task 4: Caching Data

- This simple example shows how to create an RDD and cache it. 
- Notice the 10x speed improvement! 
- If you wish to see the actual computation time, browse to the Spark UI...
- it's at host:4040. You'll see that the second calculation took much less time!

In [17]:
import time 

test = sc.parallelize(range(1,50000),4)
test.cache()

t1 = time.time()
# first count will trigger evaluation of count *and* cache
count1 = test.count()
dt1 = time.time() - t1
print("dt1: ", dt1)


t2 = time.time()
# second count operates on cached data only
count2 = test.count()
dt2 = time.time() - t2
print("dt2: ", dt2)

dt1:  0.2404930591583252
dt2:  0.03171801567077637
